# Convert balanced h5ad to parquet and split

In [1]:
import os
from deeptan.utils.data import (
    read_h5ad,
    h5ad_to_parquet,
    JointStratifiedSplitter,
)

/home/wuch/miniforge3/envs/t1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Start

In [2]:
seeds = [i + 42 for i in range(5)]

### GSE226097 ath_snrna_6tissues_Epidermal.balanced

In [3]:
path_h5ad = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/snRNA/ath_snrna_6tissues_Epidermal.balanced.h5ad"

In [4]:
path_parquet = path_h5ad.replace(".h5ad", ".parquet")
if not os.path.exists(path_parquet):
    h5ad_to_parquet(path_h5ad, path_parquet, True)

Read /mnt/hdd2/homext/wuch/xn2p/data/raw_df/snRNA/ath_snrna_6tissues_Epidermal.balanced.h5ad with 17118 cells and 18400 features.
Saving to /mnt/hdd2/homext/wuch/xn2p/data/raw_df/snRNA/ath_snrna_6tissues_Epidermal.balanced.parquet...
X shape: (17118, 18400)
DataFrame shape: (17118, 18401)
Head of DataFrame:
shape: (5, 18_401)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ obs_names ┆ AT1G01010 ┆ AT1G01020 ┆ AT1G01030 ┆ … ┆ AT5G02895 ┆ AT5G42965 ┆ AT5G44585 ┆ AT5G6362 │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ 5        │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ TCCCACATC ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 

In [5]:
celltypes = read_h5ad(path_h5ad).obs["CellType"].to_list()
batch_id = read_h5ad(path_h5ad).obs["batch"].to_list()

print(f"Number of cell types: {len(set(celltypes))}")
print(f"{set(celltypes)}")
print(f"Number of batches: {len(set(batch_id))}")
print(f"{set(batch_id)}")

Number of cell types: 6
{'flower_Epidermal', 'seedling_Epidermal', 'seed_Epidermal', 'rosette_Epidermal', 'stem_Epidermal', 'silique_Epidermal'}
Number of batches: 34
{'silique-7', 'silique-6', 'flower-6', 'rosette-2', 'stem-2', 'silique-2', 'seed-4', 'silique-8', 'silique-5', 'flower-3', 'flower-1', 'rosette-6', 'seedling-3', 'silique-1', 'seed-2', 'rosette-1', 'stem-4', 'rosette-4', 'flower-4', 'seedling-4', 'stem-1', 'seed-1', 'flower-5', 'seedling-5', 'silique-3', 'silique-4', 'seedling-2', 'seedling-6', 'stem-3', 'flower-2', 'seedling-1', 'rosette-5', 'rosette-3', 'seed-3'}


In [6]:
def _tmp_run(_ratio, _seeds, _suff: str):
    output_dir = path_parquet.replace(".parquet", "") + ".split" + "_" + _suff

    _splitter = JointStratifiedSplitter(
        cell_types=celltypes,
        orig_idents=batch_id,
        parquet_file=path_parquet,
        output_dir=output_dir,
        ratio=_ratio,
        seeds=_seeds,
        balance_strategy="none",
    )

    _splitter.execute()                                                                                                                                                                             

In [7]:
_tmp_run([0.8] + [0.1, 0.1], seeds, "full")
_tmp_run([0.4, 0.4] + [0.1, 0.1], seeds, "half")
_tmp_run([0.2, 0.2, 0.2, 0.2] + [0.1, 0.1], seeds, "quarter")
_tmp_run([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1] + [0.1, 0.1], seeds, "eighth")

Created split 0 (seed 42) with 13694 samples
Created split 1 (seed 42) with 1710 samples
Created split 2 (seed 42) with 1714 samples
Created split 0 (seed 43) with 13694 samples
Created split 1 (seed 43) with 1710 samples
Created split 2 (seed 43) with 1714 samples
Created split 0 (seed 44) with 13694 samples
Created split 1 (seed 44) with 1710 samples
Created split 2 (seed 44) with 1714 samples
Created split 0 (seed 45) with 13694 samples
Created split 1 (seed 45) with 1710 samples
Created split 2 (seed 45) with 1714 samples
Created split 0 (seed 46) with 13694 samples
Created split 1 (seed 46) with 1710 samples
Created split 2 (seed 46) with 1714 samples
Created split 0 (seed 42) with 6849 samples
Created split 1 (seed 42) with 6849 samples
Created split 2 (seed 42) with 1710 samples
Created split 3 (seed 42) with 1710 samples
Created split 0 (seed 43) with 6849 samples
Created split 1 (seed 43) with 6849 samples
Created split 2 (seed 43) with 1710 samples
Created split 3 (seed 43) w

### 独立测试

In [4]:
path_h5ad = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/snRNA/ath_snrna_6tissues_Epidermal.balanced.indep_test.h5ad"
path_parquet = path_h5ad.replace(".h5ad", ".parquet")
if not os.path.exists(path_parquet):
    h5ad_to_parquet(path_h5ad, path_parquet, True)

In [7]:
import polars as pl

In [5]:
dir_extra = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/snRNA/ath_snrna_6tissues_Epidermal.balanced.split_full.extra"
if not os.path.exists(dir_extra):
    os.makedirs(dir_extra)

In [8]:
path_extra_test = "/mnt/hdd2/homext/wuch/xn2p/data/raw_df/snRNA/ath_snrna_6tissues_Epidermal.balanced.indep_test.parquet"
df_extra_test = pl.read_parquet(path_extra_test)
print(df_extra_test)

shape: (33_077, 18_401)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ obs_names ┆ AT1G01010 ┆ AT1G01020 ┆ AT1G01030 ┆ … ┆ AT5G02895 ┆ AT5G42965 ┆ AT5G44585 ┆ AT5G6362 │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ 5        │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AAAGGTATC ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ TAGTACG-1 ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ _1-flower ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ AACAGGGAG ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0

In [9]:
# for _s in seeds:
#     _path = os.path.join(dir_extra, f"split_{_s}_2.parquet")
#     df_test = pl.read_parquet(_path)
#     print(df_test.shape)
#     df_test = pl.concat([df_test, df_extra_test])
#     print(df_test.shape, "\n")
#     df_test.write_parquet(_path)

(1714, 18401)
(34791, 18401) 

(1714, 18401)
(34791, 18401) 

(1714, 18401)
(34791, 18401) 

(1714, 18401)
(34791, 18401) 

(1714, 18401)
(34791, 18401) 

